# YZM 513 — Week 9 Lab: The CUDA Memory Hierarchy and Profiling

**İstanbul Medeniyet Üniversitesi — AI Engineering M.Sc. (Fall 2026)**

Six experiments, one story: a kernel is fast when **each byte fetched from VRAM is used many times**.
You will measure coalescing, tile a matmul, break it on purpose, see a bank conflict, write a block reduction, say which ceiling your kernel is near, and finally hide copies behind compute with streams.

**Rules for every experiment**

1. **Predict first.** Write your guess in the answer cell *before* you run the script.
2. **Verify, then time.** Every script checks its result against NumPy or SciPy before it reports a time.
3. **State numbers, not adjectives.** "2.3× faster" or "74 GB/s of 320", not "much faster".

The deck is `lectures/09memory-hierarchy-and-profiling.md`; each experiment names the part it belongs to.

## 0. Setup

In Colab select **Runtime → Change runtime type → T4 GPU** before running the next cells.
The command below targets CUDA 12. If your runtime uses CUDA 13, use `numba-cuda[cu13]` and the matching CuPy package
(see the [Numba CUDA installation guide](https://nvidia.github.io/numba-cuda/user/installation.html)).

In [ ]:
!python -m pip install -q "numba-cuda[cu12]" cupy-cuda12x

In [ ]:
!git clone -q https://github.com/adaskin/hpc-for-ai-course 2>/dev/null || true
%cd /content/hpc-for-ai-course/lectures/week09

In [ ]:
from numba import cuda
from gpu_utils import device_report

assert cuda.is_available(), "Enable an NVIDIA GPU runtime before continuing."
device_report();

Write down the GPU you were given. Every ceiling below (320 GB/s, 8,100 GFLOP/s) is for a **T4**; if you have a different GPU, look up its peak bandwidth and pass it with `--peak`.

| Question | Your answer |
|---|---|
| GPU name | |
| SM count | |
| Max shared memory per SM (bytes) | |
| Max threads per SM | |

## Experiment 1 — Coalescing (Part III)

`02_coalescing.py` sums a 4096 × 4096 float32 matrix two ways: **one thread per row** and **one thread per column**.
Both read every element once and do the same arithmetic.

**Predict:** which kernel is faster, and by what factor? Why?

> Your prediction:

In [ ]:
!python 02_coalescing.py

| kernel | ms | achieved GB/s | % of 320 GB/s |
|---|---:|---:|---:|
| `col_sums` (thread per column) | | | |
| `row_sums` (thread per row) | | | |

**Questions**

1. At one step of the loop, which addresses do the 32 lanes of a warp read in `row_sums`? In `col_sums`? (Use `A[i, c]` and the row length N.)
2. The sector model predicts up to 8× more traffic for the strided kernel. Is your measured ratio larger or smaller? What could explain the difference?
3. Re-run with `--n 2048`. Does the gap change?

> Your answers:

## Experiment 2 — Naive vs tiled matmul (Parts I, IV, VI)

First the model, which needs no GPU, then the measurements.

In [ ]:
!python 01_traffic_accounting.py

In [ ]:
!python 05_tile_sweep.py --sizes 512 1024 2048

| N | version | ms | GFLOP/s | blocks/SM | occupancy |
|---:|---|---:|---:|---:|---:|
| 1024 | naive | | | | |
| 1024 | tiled T=8 | | | | |
| 1024 | tiled T=16 | | | | |
| 1024 | tiled T=32 | | | | |
| 1024 | cuBLAS (CuPy) | | | | |

**Questions**

1. Naive at N = 1024 must read 8.6 GB. At 320 GB/s that is 27 ms. How does your measured time compare? What does that tell you about the caches?
2. Tiling with T = 16 divides global traffic by 16. Is the kernel 16× faster? What else limits it now?
3. T = 8, 16 and 32 all reach 100 % theoretical occupancy. Why do their runtimes still differ?
4. How far is your best tiled kernel from cuBLAS? Name two things cuBLAS does that ours does not.

> Your answers:

## Experiment 3 — Remove a barrier (Part IV)

The tiled kernel has two `cuda.syncthreads()` calls per tile. `--race` runs two broken variants three times each.

**Predict:** will removing a barrier crash the kernel? Will the answer be wrong every time?

> Your prediction:

In [ ]:
!python 04_matmul_tiled.py --race

| variant | run 0 max err | run 1 max err | run 2 max err |
|---|---|---|---|
| drop barrier after the load | | | |
| drop barrier after the compute | | | |

**Questions**

1. Which interleaving of two threads makes each variant wrong? Describe it in one sentence each.
2. Did any run look correct? What would that mean for a test suite that checks one run?
3. Which barrier can you remove safely? Justify it with your three runs.

> Your answers:

## Experiment 4 — Bank conflicts in a transpose (Part IV)

The tiled matmul does **not** have bank conflicts (its inner reads are broadcasts or conflict-free). The transpose does: the shared read `tile[tx, ty]` puts all 32 lanes in the same bank when the tile is 32 × 32. Padding the tile to 32 × 33 spreads them over 32 banks.

**Predict:** how much faster is the padded version?

> Your prediction:

In [ ]:
!python 06_bank_conflicts.py

| version | shared KB | ms | GB/s |
|---|---:|---:|---:|
| unpadded (32, 32) | | | |
| padded (32, 33) | | | |

**Questions**

1. Compute by hand the bank of lane `tx` for `ty = 5` in both versions. Do you get 1 distinct bank and 32 distinct banks?
2. Does padding help on your GPU? If the gain is small, say so and suggest why (hint: what else does this kernel spend time on?).

> Your answers:

## Experiment 5 — Reductions, softmax, and profiling (Parts V and VII)

Block reduction in shared memory, then Week 8's softmax redone with cooperating threads.

In [ ]:
!python 07_block_reduce.py

In [ ]:
!python 08_softmax_shared.py

**Questions**

1. How many atomic operations does each version of the sum issue for n = 2²⁴? Which is faster, and by what factor?
2. The two sums have different last digits. Why? Which one is closer to the float64 reference?
3. Why must `cuda.syncthreads()` sit *outside* the `if tid < stride:` in the reduction loop?
4. The third softmax case has 37 columns and 64 threads. What do the padding lanes contribute to the max? To the sum?

> Your answers:

Now profile. `09_profile_and_measure.py` tries Nsight Compute (`ncu`) first and falls back to measurement when it is missing or when counters are blocked (`ERR_NVGPUCTRPERM`).

In [ ]:
!which ncu || echo "ncu not installed"

In [ ]:
!python 09_profile_and_measure.py --n 1024

Optional: the `torch.profiler` timeline (Colab has PyTorch). Download `week09_torch_trace.json` and open it at <https://ui.perfetto.dev>.

In [ ]:
!python 09_profile_and_measure.py --n 1024 --no-ncu --torch

## Experiment 6 — Streams: overlap copies with compute (Part VIII)

`10_streams_overlap.py` runs an arithmetic-heavy elementwise kernel two ways: upload everything → kernel → download everything, and K chunks each in its own stream with pinned host memory.

**Predict:** for K = 4 balanced stages the ideal speedup is 3K / (K + 2) = 2×. Will you reach it? What happens with `--iters 1`, and with `--chunks 1`?

> Your prediction:

In [ ]:
!python 10_streams_overlap.py --chunks 4

In [ ]:
!python 10_streams_overlap.py --chunks 8
!python 10_streams_overlap.py --chunks 4 --iters 1

| chunks | iterations per element | serial ms | streams ms | speedup |
|---:|---:|---:|---:|---:|
| 4 | 200 | | | |
| 8 | 200 | | | |
| 4 | 1 | | | |

**Questions**

1. Why is the speedup below the ideal 3K / (K + 2)? Name two reasons.
2. Why does a light kernel (`--iters 1`) gain almost nothing?
3. The script allocates the host arrays with `cuda.pinned_array`. What goes wrong if you use ordinary `np.empty` instead? Try it and report what you see.

> Your answers:

## Answer sheet

For your best tiled kernel at N = 1024:

1. **Bytes:** how many bytes did it move from VRAM (model)? Show the arithmetic.
2. **Intensity:** FLOPs ÷ bytes. Where does that sit relative to the T4 ridge point (about 25 FLOP/byte)?
3. **What changed** from the naive kernel: did tiling reduce *traffic*, raise *occupancy*, or both?
4. **The sentence:** *"My kernel is ______-bound, because it reaches ______ GB/s of 320 and ______ GFLOP/s of 8,100."*

> Your answers:

## Bonus

Replace the last five rounds of the shared-memory reduction (`stride` = 16, 8, 4, 2, 1) with warp shuffles, `cuda.shfl_down_sync(0xffffffff, val, delta)`, so the last 32 lanes exchange registers instead of shared memory. Verify against NumPy, then time it against `07_block_reduce.py`.

## Submit

This notebook with your outputs, the completed tables, your answers, and the GPU model from the setup cell.